# LAB | RAG with the training wheels off

## Step 1: Setup and Document Preparation

In [1]:
pip install openai numpy python-dotenv pypdf tiktoken -q


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
print("Key loaded:", bool(os.getenv("OPENAI_API_KEY")))

Key loaded: True


In [ ]:
# Load all source documents from the "input" folder
import os
from pypdf import PdfReader

def load_pdf(path: str) -> str:
    """Extract raw text from a single PDF file."""
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def load_documents_from_folder(folder: str) -> dict[str, str]:
    """
    Load every PDF in a folder.
    Returns a dict mapping filename -> raw text,
    so we can later trace a chunk back to its source document.
    """
    documents = {}
    for filename in os.listdir(folder):
        if filename.endswith(".pdf"):
            path = os.path.join(folder, filename)
            documents[filename] = load_pdf(path)
    return documents

docs = load_documents_from_folder("input")
for name, text in docs.items():
    print(f"{name}: {len(text)} characters")

ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf: 156411 characters
385082eng.pdf: 22103 characters


In [ ]:
# Split each document into overlapping chunks, keeping track of source
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """
    Split text into overlapping chunks by character count.
    Overlap prevents a relevant sentence from being cut in half
    right at a chunk boundary.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap  # move forward, keeping the overlap region
    return chunks

# all_chunks stores (source_document, chunk_text) tuples
all_chunks = []
for filename, text in docs.items():
    for chunk in chunk_text(text):
        all_chunks.append((filename, chunk))

print(f"Total chunks across all documents: {len(all_chunks)}")
print(f"Example chunk from '{all_chunks[0][0]}':")
print(all_chunks[0][1][:200])

Total chunks across all documents: 398
Example chunk from 'ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf':
INDEPENDENT 
HIGH-LEVEL EXPERT GROUP ON 
ARTIFICIAL INTELLIGENCE 
SET UP BY THE EUROPEAN COMMISSION 
 
 
 
 
 
 
ETHICS GUIDELINES 
FOR TRUSTWORTHY AI 
 
 
 
 
 
 
ETHICS GUIDELINES FOR TRUSTWORTHY AI


## Step 2: Generate Embeddings

In [ ]:
# Generate embeddings in batches instead of one-by-one
def get_embeddings_batch(texts: list[str], model: str = "text-embedding-3-small") -> list[list[float]]:
    """Call the OpenAI embeddings endpoint for a batch of texts at once."""
    response = client.embeddings.create(input=texts, model=model)
    # Results come back in the same order as the input texts
    return [item.embedding for item in response.data]

batch_size = 100
chunk_embeddings = []
texts_only = [chunk for source, chunk in all_chunks]

for i in range(0, len(texts_only), batch_size):
    batch = texts_only[i:i + batch_size]
    chunk_embeddings.extend(get_embeddings_batch(batch))
    print(f"Embedded {min(i + batch_size, len(texts_only))}/{len(texts_only)} chunks")

print(f"Embedding dimension: {len(chunk_embeddings[0])}")

Embedded 100/398 chunks
Embedded 200/398 chunks
Embedded 300/398 chunks
Embedded 398/398 chunks
Embedding dimension: 1536


In [9]:
# Single-text embedding helper
def get_embedding(text: str, model: str = "text-embedding-3-small") -> list[float]:
    """Call the OpenAI embeddings endpoint for a single piece of text (e.g. a user query)."""
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

## Step 3: Implement Vector Search

In [7]:
# Compute cosine similarity and retrieve the top-k most relevant chunks
import numpy as np

def cosine_similarity(a, b):
    """Cosine similarity between two vectors — measures angle, not magnitude."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve_top_k(query: str, k: int = 3):
    """
    Embed the query, compare it against every stored chunk embedding,
    and return the top-k most similar chunks along with their source
    document and similarity score (needed for the lab_proof.md evidence).
    """
    query_embedding = get_embedding(query)

    scored = []
    for (source, chunk), chunk_embedding in zip(all_chunks, chunk_embeddings):
        score = cosine_similarity(query_embedding, chunk_embedding)
        scored.append((score, source, chunk))

    # Sort by similarity score, descending
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:k]

In [10]:
# Checkpoint — test with a sample query
results = retrieve_top_k("What are the requirements for human oversight of AI systems?", k=3)

for score, source, chunk in results:
    print(f"Score: {score:.4f} | Source: {source}")
    print(chunk[:200])
    print("---")

Score: 0.7094 | Source: ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf
to be subject to a decision based solely on automated processing when this 
produces legal effects on users or similarly significantly affects them.36 
Human oversight. Human oversight helps ensuring 
---
Score: 0.7027 | Source: ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf
gations, both as regards horizontally applicable rules as well as domain -specific 
regulation. 
In the following paragraphs, each requirement is explained in more detail.  
 
1.1 Human agency and ove
---
Score: 0.6715 | Source: ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf
es? 
 Did you take safeguards to prevent overconfidence in or overreliance on the AI system for work 
processes? 
Human oversight: 
 Did you consider the appropriate level of human control for the p
---


In [11]:
# Test with an off-topic query for comparison
results_offtopic = retrieve_top_k("What is the best recipe for lasagna?", k=3)

for score, source, chunk in results_offtopic:
    print(f"Score: {score:.4f} | Source: {source}")
    print(chunk[:150])
    print("---")

Score: 0.1295 | Source: ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf
lues and (3) it should be r obust, 
both from a technical and social perspective since to ensure that, even with good intentions, AI systems do not 
c
---
Score: 0.1205 | Source: ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf
group-artificial-intelligence).  
The reuse policy of European Commission documents is regulated by Decision 2011/8 33/EU (OJ L 330, 14. 12.2011, p.39
---
Score: 0.1183 | Source: ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf
and New Technologies (“EGE”) 
proposed a set of 9 basic principles, based on the fundamental values laid down in the EU Treaties and Charter. 24 
We b
---


## Step 4: Build RAG Query Function

In [12]:
# Build the full RAG pipeline — retrieve context, then generate an answer
def rag_query(question: str, k: int = 3) -> dict:
    """
    Full RAG pipeline: retrieve top-k chunks, build a grounded prompt,
    and generate an answer. Returns a trace dict capturing every step,
    so the answer can be traced back to its evidence for lab_proof.md.
    """
    top_chunks = retrieve_top_k(question, k=k)
    context = "\n\n".join(f"[{source}]\n{chunk}" for score, source, chunk in top_chunks)

    prompt = f"""Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}
Answer:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    answer = response.choices[0].message.content

    # This trace is the proof-of-grounding evidence for lab_proof.md
    trace = {
        "query": question,
        "retrieved_chunks": [
            {"source": source, "score": round(float(score), 4), "text": chunk}
            for score, source, chunk in top_chunks
        ],
        "final_answer": answer,
    }
    return trace

In [13]:
# Checkpoint — test with your grounded question
trace_success = rag_query("What are the requirements for human oversight of AI systems?")
print("ANSWER:")
print(trace_success["final_answer"])

ANSWER:
The requirements for human oversight of AI systems include:

1. Ensuring that AI systems support human autonomy and decision-making.
2. Implementing governance mechanisms such as human-in-the-loop (HITL), human-on-the-loop (HOTL), or human-in-command (HIC) approaches to allow for human intervention.
3. Considering the appropriate level of human control for the particular AI system and use case.
4. Describing the level of human control or involvement, including identifying who is the “human in control” and the moments or tools for human intervention.
5. Establishing mechanisms and measures to ensure human control or oversight.


In [14]:
# Checkpoint — test with your off-topic question (the failure case)
trace_failure = rag_query("What is the best recipe for lasagna?")
print("ANSWER:")
print(trace_failure["final_answer"])

ANSWER:
I don't know.


## Create Lab Proof

In [15]:
# Assemble lab_proof.md from the two traces captured above
def format_trace_as_markdown(trace: dict, label: str) -> str:
    """Format one trace (query + evidence + answer) as a markdown section."""
    lines = [f"## {label}", "", f"**Query:** {trace['query']}", "", "**Retrieved evidence:**", ""]
    for i, chunk in enumerate(trace["retrieved_chunks"], 1):
        lines.append(f"{i}. Score: {chunk['score']} | Source: `{chunk['source']}`")
        lines.append(f"   > {chunk['text'][:200]}...")
        lines.append("")
    lines.append(f"**Final answer:** {trace['final_answer']}")
    lines.append("")
    return "\n".join(lines)

proof_content = f"""# Lab Proof — RAG with the Training Wheels Off

## Pipeline overview

- Loaded 2 source PDFs from `input/` (EU HLEG Ethics Guidelines, UNESCO AI Ethics Recommendation)
- Split into {len(all_chunks)} overlapping chunks (chunk_size=500, overlap=50 characters)
- Generated embeddings using OpenAI's `text-embedding-3-small` (1536 dimensions)
- Retrieval via cosine similarity, top-k=3
- Answer generation via `gpt-4o-mini`, grounded strictly in retrieved context

{format_trace_as_markdown(trace_success, "Case 1: Successful grounded retrieval")}

{format_trace_as_markdown(trace_failure, "Case 2: Failure case (out-of-scope query)")}

## Grounding risk / limitation

The system produced a plausible but potentially unsupported answer risk mainly
around **corpus imbalance**: the EU HLEG document accounts for ~87% of all
chunks (156k of 178k total characters), so retrieval is structurally biased
toward it even when the UNESCO document contains equally relevant content.
A query whose best answer lives primarily in the smaller document could
retrieve HLEG chunks with moderate similarity instead of the more relevant
(but underrepresented) UNESCO passage — producing an answer that sounds
grounded but omits the more accurate source.

The out-of-scope test (Case 2) confirms the system correctly refuses to
answer when no relevant evidence is retrieved (similarity scores ~0.12 vs.
~0.70 for the grounded case), rather than hallucinating a plausible-sounding
answer from unrelated context.
"""

with open("lab_proof.md", "w", encoding="utf-8") as f:
    f.write(proof_content)

print("lab_proof.md written successfully")
print(f"Length: {len(proof_content)} characters")

lab_proof.md written successfully
Length: 4250 characters
